# Lecture 1 — RAG Foundations

**Course:** RAG Engineering — From Retrieval to Production

---

## What this notebook covers

| # | Topic |
|---|---|
| 1 | Generative AI fundamentals |
| 2 | LLM limitations |
| 3 | Knowledge cutoff |
| 4 | Hallucination |
| 5 | Context-window limitations |
| 6 | Why RAG? |
| 7 | RAG vs fine-tuning |
| 8 | RAG vs long-context LLMs |
| 9 | RAG architecture |
| 10 | RAG pipeline lifecycle |
| 11 | Indexing pipeline |
| 12 | Retrieval pipeline |
| 13 | Generation pipeline |
| 14 | Grounding and citations |
| 15 | Enterprise use cases |

## How to run it

```bash
pip install numpy pandas matplotlib scikit-learn openai langchain-core langchain-openai
```

That is all you need. **The notebook runs end to end with no API key and no internet.**

Where a real embedding model or LLM would be used, it falls back to a small local
stand-in so the mechanics stay visible. If you *do* have a key, set these before
starting and the same code calls the real thing:

```bash
export EURI_API_KEY="..."
export EURI_CHAT_MODEL="gpt-4.1-mini"
export EURI_EMBEDDING_MODEL="<embedding model enabled on your account>"
pip install openai
```

> **Teaching principle:** do not start with a framework. Start with the problem each piece solves.

In [10]:
!pip install numpy pandas matplotlib scikit-learn openai langchain-core langchain-openai


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# Setup — run this first
import os, math, textwrap, re
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# --- optional live backend (EURI). Everything works without it. -------------
EURI_BASE_URL        = "https://api.euron.one/api/v1/euri"
EURI_API_KEY         = os.getenv("EURI_API_KEY")
EURI_CHAT_MODEL      = os.getenv("EURI_CHAT_MODEL", "gpt-4.1-mini")
EURI_EMBEDDING_MODEL = os.getenv("EURI_EMBEDDING_MODEL")
EURI_EMBEDDING_DIMENSION = int(os.getenv("EURI_EMBEDDING_DIMENSION", 1536))

try:
    from openai import OpenAI
    _OPENAI_SDK = True
except ImportError:
    _OPENAI_SDK = False

LIVE_LLM   = bool(EURI_API_KEY and _OPENAI_SDK)
LIVE_EMBED = bool(EURI_API_KEY and EURI_EMBEDDING_MODEL and _OPENAI_SDK)

print("openai sdk installed :", _OPENAI_SDK)
print("live LLM calls       :", LIVE_LLM,   "" if LIVE_LLM   else "-> using a scripted stand-in")
print("live embeddings      :", LIVE_EMBED, "" if LIVE_EMBED else "-> using a local TF-IDF stand-in")
print("embedding dimension  :", EURI_EMBEDDING_DIMENSION)

openai sdk installed : True
live LLM calls       : True 
live embeddings      : True 
embedding dimension  : 1536


## Euri Embedding Model Setup

In [12]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=EURI_EMBEDDING_MODEL,
    api_key=EURI_API_KEY,
    base_url=EURI_BASE_URL,
    check_embedding_ctx_length=False,
    dimensions=EURI_EMBEDDING_DIMENSION
)

print("Embedding model configured:", EURI_EMBEDDING_MODEL)

Embedding model configured: gemini-embedding-2-preview


In [13]:
test_vector = embeddings.embed_query(
    "What is Corrective Retrieval-Augmented Generation?"
)

print("Embedding dimension:", len(test_vector))

Embedding dimension: 1536


In [14]:
test_vector

[-0.0024213625583797693,
 -0.003234507981687784,
 0.008675139397382736,
 0.014268480241298676,
 -0.012205798178911209,
 -0.016287043690681458,
 -0.006864191498607397,
 0.0036184273194521666,
 -0.013394001871347427,
 -0.05471065640449524,
 -0.02056209370493889,
 0.003130154451355338,
 0.009612931869924068,
 -0.0022360417060554028,
 0.01125081442296505,
 -0.011899606324732304,
 0.05366623401641846,
 -0.008887841366231441,
 -0.019573360681533813,
 -0.03550248593091965,
 -0.02695542387664318,
 0.013943742029368877,
 0.023841461166739464,
 0.012781135737895966,
 -0.003394970204681158,
 0.005004988983273506,
 0.019234925508499146,
 -0.013474741019308567,
 0.03270634636282921,
 0.16843293607234955,
 -0.03381215035915375,
 -0.04515211656689644,
 -0.01814260147511959,
 -0.03873325139284134,
 0.013752066530287266,
 0.0285735335201025,
 0.016116896644234657,
 0.030141592025756836,
 0.016381051391363144,
 0.03924043849110603,
 0.024645088240504265,
 -0.009015683084726334,
 -0.016713758930563927,
 

## Euri Chat Model Setup

In [15]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=EURI_CHAT_MODEL,
    api_key=EURI_API_KEY,
    base_url=(
        "https://api.euron.one/"
        "api/v1/euri"
    ),
    temperature=0.2,
    max_completion_tokens=1000
)

In [16]:
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="What is Machine Learning")])
response

AIMessage(content='Machine Learning is a branch of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to learn and make decisions or predictions based on data, without being explicitly programmed for specific tasks. Instead of following fixed instructions, machine learning systems improve their performance by identifying patterns and relationships within data.\n\nKey points about Machine Learning:\n- **Data-driven:** It relies on large amounts of data to train models.\n- **Learning from experience:** Models improve over time as they are exposed to more data.\n- **Types:** Common types include supervised learning (learning from labeled data), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning through trial and error with feedback).\n- **Applications:** Used in various fields such as image and speech recognition, recommendation systems, natural language processing, autonomous vehicl

In [17]:
response.content

'Machine Learning is a branch of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to learn and make decisions or predictions based on data, without being explicitly programmed for specific tasks. Instead of following fixed instructions, machine learning systems improve their performance by identifying patterns and relationships within data.\n\nKey points about Machine Learning:\n- **Data-driven:** It relies on large amounts of data to train models.\n- **Learning from experience:** Models improve over time as they are exposed to more data.\n- **Types:** Common types include supervised learning (learning from labeled data), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning through trial and error with feedback).\n- **Applications:** Used in various fields such as image and speech recognition, recommendation systems, natural language processing, autonomous vehicles, and more.\n\nI

---
# 1. Generative AI fundamentals

An LLM does exactly one thing, over and over: **given the text so far, predict the next token.**

$$P(x_t \mid x_1, x_2, \ldots, x_{t-1})$$

Everything else — answering, summarising, writing code — is that one operation repeated.

Two consequences matter for us:

1. **Fluency is not truth.** The model optimises for *likely* text, not *correct* text. A confident sentence is evidence of nothing.
2. **Knowledge lives in the weights.** Facts are smeared across billions of parameters. There is no row to look up, no source to cite, no record to update.

Let's make next-token prediction concrete with a tiny model we can actually inspect.

In [18]:
# A 2-gram "language model" — the smallest thing that still counts as one.
from collections import defaultdict, Counter

corpus = """
the support team answers refund questions .
the support team answers billing questions .
the support team escalates security questions .
the finance team answers billing questions .
"""

tokens = corpus.split()
bigram = defaultdict(Counter)
for a, b in zip(tokens, tokens[1:]):
    bigram[a][b] += 1

def next_token_distribution(word):
    counts = bigram[word]
    total  = sum(counts.values())
    if total == 0:
        return pd.DataFrame(columns=["next_token", "P"])
    rows = [{"next_token": t, "P": round(c / total, 3)} for t, c in counts.most_common()]
    return pd.DataFrame(rows)

print('After the word "team", the model believes:')
display(next_token_distribution("team"))
print('After the word "answers", the model believes:')
display(next_token_distribution("answers"))

After the word "team", the model believes:


,next_token,P
0,answers,0.75
1,escalates,0.25


After the word "answers", the model believes:


,next_token,P
0,billing,0.667
1,refund,0.333


In [19]:
tokens

['the',
 'support',
 'team',
 'answers',
 'refund',
 'questions',
 '.',
 'the',
 'support',
 'team',
 'answers',
 'billing',
 'questions',
 '.',
 'the',
 'support',
 'team',
 'escalates',
 'security',
 'questions',
 '.',
 'the',
 'finance',
 'team',
 'answers',
 'billing',
 'questions',
 '.']

In [20]:
bigram

defaultdict(collections.Counter,
            {'the': Counter({'support': 3, 'finance': 1}),
             'support': Counter({'team': 3}),
             'team': Counter({'answers': 3, 'escalates': 1}),
             'answers': Counter({'billing': 2, 'refund': 1}),
             'refund': Counter({'questions': 1}),
             'questions': Counter({'.': 4}),
             '.': Counter({'the': 3}),
             'billing': Counter({'questions': 2}),
             'escalates': Counter({'security': 1}),
             'security': Counter({'questions': 1}),
             'finance': Counter({'team': 1})})

In [21]:
bigram["team"]

Counter({'answers': 3, 'escalates': 1})

Notice what just happened. The model has **no idea** what a refund is. It learned that
`team` is usually followed by `answers`. Scale that from 4 sentences to trillions of tokens and
you get something that sounds expert — but the mechanism has not changed.

## Parametric vs external knowledge

| | Parametric knowledge | External knowledge |
|---|---|---|
| Where it lives | inside the model weights | in your documents, databases, APIs |
| When it was fixed | at training time | right now |
| Can you update one fact? | no — retrain | yes — edit the source |
| Can you delete a fact? | not reliably | yes |
| Can you cite it? | no | yes |
| Can you permission it? | no | yes |

**RAG is the bridge between the two columns.** Let's simulate both stores.

In [22]:
# Two knowledge stores: one frozen in the weights, one live in the business.

PARAMETRIC_MEMORY = {                    # what the model "learned" during training
    "cancellation_window_days": 30,      # true in 2024
    "support_hours": "9am-6pm IST",
    "_frozen_at": "2024-06",
}

EXTERNAL_STORE = {                       # what is actually true today, in your systems
    "cancellation_window_days": 14,      # policy changed in 2026
    "support_hours": "24x7",
    "_last_updated": "2026-09-01",
}

comparison = pd.DataFrame(
    [{"fact": k,
      "model believes": PARAMETRIC_MEMORY[k],
      "actually true": EXTERNAL_STORE[k],
      "agrees?": "yes" if PARAMETRIC_MEMORY[k] == EXTERNAL_STORE[k] else "NO"}
     for k in ("cancellation_window_days", "support_hours")]
)
display(comparison)
print("\nThe model is not lying. It is answering correctly — about June 2024.")

,fact,model believes,actually true,agrees?
0,cancellation_window_days,30,14,NO
1,support_hours,9am-6pm IST,24x7,NO



The model is not lying. It is answering correctly — about June 2024.


---
# 2. LLM limitations

Every one of these is a reason RAG exists. Learn them as a checklist: when a RAG system
misbehaves in production, the cause is almost always one of these.

In [23]:
limitations = pd.DataFrame([
 ["Knowledge cutoff", "Facts stop at the training date",
  "Quotes last year's price or policy", "Retrieve the current approved source"],
 ["Hallucination", "Fluent output that nothing supports",
  "Invented refund clause reaches a customer", "Ground generation in supplied evidence"],
 ["No private knowledge", "Your internal docs were never in training",
  "Cannot answer anything company-specific", "Retrieve from your own corpus"],
 ["Context-window limit", "Only N tokens fit in one request",
  "Cannot paste a 40,000-document corpus", "Retrieve a small, relevant subset"],
 ["No provenance", "Parametric answers have no source",
  "Nobody can audit or verify the answer", "Carry metadata and citations through"],
 ["No access control", "The model does not know who may see what",
  "Finance doc leaks into an HR answer", "Filter retrieval by role and tenant"],
 ["Cost and latency", "Every token is paid for, twice over",
  "Long prompts are slow and expensive", "Send the smallest sufficient evidence"],
], columns=["Limitation", "What it means", "What breaks in production", "What RAG does about it"])

display(limitations)
print("Note: RAG does not delete these problems. It gives you a place to fix each one.")

,Limitation,What it means,What breaks in production,What RAG does about it
0,Knowledge cutoff,Facts stop at the training date,Quotes last year's price or policy,Retrieve the current approved source
1,Hallucination,Fluent output that nothing supports,Invented refund clause reaches a customer,Ground generation in supplied evidence
2,No private knowledge,Your internal docs were never in training,Cannot answer anything company-specific,Retrieve from your own corpus
3,Context-window limit,Only N tokens fit in one request,"Cannot paste a 40,000-document corpus","Retrieve a small, relevant subset"
4,No provenance,Parametric answers have no source,Nobody can audit or verify the answer,Carry metadata and citations through
5,No access control,The model does not know who may see what,Finance doc leaks into an HR answer,Filter retrieval by role and tenant
6,Cost and latency,"Every token is paid for, twice over",Long prompts are slow and expensive,Send the smallest sufficient evidence


Note: RAG does not delete these problems. It gives you a place to fix each one.


---
# 3. Knowledge cutoff

> *"What is our current Enterprise Annual cancellation policy?"*

A plain LLM has three options, and all three are bad:

1. answer from stale training data → **confidently wrong**
2. guess a plausible-sounding rule → **hallucination**
3. refuse → **unhelpful**, even though the correct answer exists in your Confluence

Freshness is not one thing. In production it has **four layers**, and any of them can be stale:

| Layer | Question | Owner |
|---|---|---|
| Source freshness | Has the document itself changed? | the business |
| Index freshness | Have we re-ingested it since? | the ingestion pipeline |
| Retrieval freshness | Can the newest version be *found*? | retrieval + filters |
| Answer freshness | Did generation actually *use* it? | context + prompt |

A system can pass layers 1–3 and still fail at 4. Let's watch that happen.

In [25]:
# Closed book vs open book, on a fact that changed.

QUESTION = "How many days do I have to cancel an Enterprise Annual plan?"

def closed_book_answer():
    """A scripted stand-in for what an LLM says from memory alone."""
    d = PARAMETRIC_MEMORY["cancellation_window_days"]
    return f"You can cancel an Enterprise Annual plan within {d} days of purchase."

def open_book_answer(evidence: str):
    """The same question, answered from supplied evidence."""
    m = re.search(r"within (\d+) calendar days", evidence)
    return (f"You can cancel within {m.group(1)} calendar days of purchase. [S1]"
            if m else "The evidence provided does not state a cancellation window.")

CURRENT_POLICY = ("Enterprise Cancellation Policy, version 2026-09. Enterprise Annual plans "
                  "may be cancelled within 14 calendar days of initial purchase.")

print("CLOSED BOOK (model memory):", closed_book_answer())
print("OPEN BOOK  (retrieved)   :", open_book_answer(CURRENT_POLICY))
print("\nSame model. Same question. The difference is entirely the evidence.")

CLOSED BOOK (model memory): You can cancel an Enterprise Annual plan within 30 days of purchase.
OPEN BOOK  (retrieved)   : You can cancel within 14 calendar days of purchase. [S1]

Same model. Same question. The difference is entirely the evidence.


---
# 4. Hallucination

"The model made it up" is not a diagnosis. These five failures look identical to a user and
need completely different fixes — that distinction is the whole job.

In [26]:
hallucination_types = pd.DataFrame([
 ["Intrinsic contradiction", "Answer contradicts the evidence you supplied",
  "Evidence says 14 days; answer says 30", "Generation / prompt"],
 ["Extrinsic fabrication", "Answer adds facts no evidence supports",
  "Invents a 'cooling-off fee'", "Generation / prompt"],
 ["Citation hallucination", "Citation points somewhere that does not support the claim",
  "Cites [S2], but [S2] is about shipping", "Citation validation"],
 ["Retrieval-induced error", "Wrong evidence retrieved, then faithfully used",
  "Retrieved the 2024 policy", "Retrieval / filtering"],
 ["Synthesis error", "Right documents, wrong combination",
  "Merges two plans' rules into one", "Context engineering"],
], columns=["Type", "What it is", "Example", "Where you fix it"])

display(hallucination_types)

,Type,What it is,Example,Where you fix it
0,Intrinsic contradiction,Answer contradicts the evidence you supplied,Evidence says 14 days; answer says 30,Generation / prompt
1,Extrinsic fabrication,Answer adds facts no evidence supports,Invents a 'cooling-off fee',Generation / prompt
2,Citation hallucination,Citation points somewhere that does not support the claim,"Cites [S2], but [S2] is about shipping",Citation validation
3,Retrieval-induced error,"Wrong evidence retrieved, then faithfully used",Retrieved the 2024 policy,Retrieval / filtering
4,Synthesis error,"Right documents, wrong combination",Merges two plans' rules into one,Context engineering


### The grounding rule

Most naive RAG prompts say "answer the question." A production prompt says something closer to:

```text
Use only the supplied evidence for company-specific claims.
If the evidence is insufficient, say so — do not fill the gap.
If sources conflict, state the conflict instead of picking one.
Cite every material claim with its source id, e.g. [S1].
```

Two of those lines matter more than people expect. **"Say so"** gives the model a legal way
out other than inventing. **"State the conflict"** stops it silently choosing the stale version.

In [27]:
GROUNDING_SYSTEM_PROMPT = """You are a grounded enterprise knowledge assistant.

Rules:
1. Use ONLY the evidence below for company-specific claims.
2. If the evidence is insufficient, say exactly: "The available sources do not answer this."
3. If sources conflict, state the conflict and cite both.
4. Cite every material factual claim with its source id, e.g. [S1].
5. Do not add information from your own training data."""

def build_prompt(question: str, evidence_blocks: List[str]) -> str:
    evidence = "\n\n".join(evidence_blocks)
    return f"""{GROUNDING_SYSTEM_PROMPT}

EVIDENCE
{evidence}

QUESTION
{question}

ANSWER (with citations):"""

demo = build_prompt(
    "What is the cancellation window for the Enterprise Annual plan?",
    ["[S1] Enterprise Cancellation Policy v2026-09: Enterprise Annual plans may be "
     "cancelled within 14 calendar days of initial purchase.",
     "[S2] Enterprise Provisioning Rules: cancellation eligibility ends once "
     "provisioning has completed."]
)
print(demo)

You are a grounded enterprise knowledge assistant.

Rules:
1. Use ONLY the evidence below for company-specific claims.
2. If the evidence is insufficient, say exactly: "The available sources do not answer this."
3. If sources conflict, state the conflict and cite both.
4. Cite every material factual claim with its source id, e.g. [S1].
5. Do not add information from your own training data.

EVIDENCE
[S1] Enterprise Cancellation Policy v2026-09: Enterprise Annual plans may be cancelled within 14 calendar days of initial purchase.

[S2] Enterprise Provisioning Rules: cancellation eligibility ends once provisioning has completed.

QUESTION
What is the cancellation window for the Enterprise Annual plan?

ANSWER (with citations):


---
# 5. Context-window limitations

Modern models take very large contexts. So why not paste the entire company?

Because the context window is a **desk, not a warehouse**. Six reasons, roughly in order of
how often they bite:

| Reason | What it costs you |
|---|---|
| Money | You pay per input token, on every single request |
| Latency | A bigger prompt is a slower response |
| Signal dilution | The right paragraph gets buried — "lost in the middle" |
| Version conflict | The 2024 and 2026 policies both arrive, and the model picks one |
| Security | Everything in the prompt is visible to the model, permissions or not |
| Debuggability | When the answer is wrong, which of 900 pages caused it? |

Let's put numbers on the first two.

In [28]:
# Rough but useful: ~4 characters per token for English prose.
def approx_tokens(text: str) -> int:
    return max(1, len(text) // 4)

# A small enterprise corpus: 8,000 documents averaging 2 pages each.
N_DOCS          = 8_000
TOKENS_PER_DOC  = 1_200
PRICE_PER_1M_IN = 0.15        # USD per 1M input tokens — set to your own model's price
QUERIES_PER_DAY = 5_000

stuff_everything = N_DOCS * TOKENS_PER_DOC
retrieve_five    = 5 * 400 + 300                     # 5 chunks + question and instructions

rows = []
for name, tokens in [("Paste the whole corpus", stuff_everything),
                     ("Retrieve 5 chunks", retrieve_five)]:
    cost_per_query = tokens / 1_000_000 * PRICE_PER_1M_IN
    rows.append({
        "approach": name,
        "input tokens / query": f"{tokens:,}",
        "cost / query (USD)": f"${cost_per_query:,.4f}",
        "cost / day (USD)": f"${cost_per_query * QUERIES_PER_DAY:,.2f}",
        "cost / year (USD)": f"${cost_per_query * QUERIES_PER_DAY * 365:,.0f}",
    })

display(pd.DataFrame(rows))
print(f"Ratio: retrieval sends {stuff_everything / retrieve_five:,.0f}x fewer tokens per query.")
print("And that is before latency, dilution, conflicts, security and debuggability.")

,approach,input tokens / query,cost / query (USD),cost / day (USD),cost / year (USD)
0,Paste the whole corpus,"9,600,000",$1.4400,"$7,200.00","$2,628,000"
1,Retrieve 5 chunks,"2,300",$0.0003,$1.72,$630


Ratio: retrieval sends 4,174x fewer tokens per query.
And that is before latency, dilution, conflicts, security and debuggability.


In [31]:
((8_000*1_200)/1_000_000)*0.15*5_000


7200.0

---
# 6. Why RAG?

RAG is an **inference-time architecture**. Nothing is retrained. At the moment a question arrives:

1. take the query
2. search an external knowledge store
3. select the evidence that matters
4. build a context out of it
5. ask the LLM to answer **using that evidence**
6. return the answer with its sources

Formally, with $q$ the query, $K$ the knowledge index, $I$ the instructions:

$$D_q = \text{Retrieve}(q, K) \qquad a = \text{LLM}(q, D_q, I)$$

The whole of the rest of this course is making `Retrieve` better and `LLM(...)` more disciplined.

Here is the entire idea in about twenty lines — no framework, no vector database yet.

In [32]:
# RAG in twenty lines, so you can see there is no magic in it.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

mini_docs = [
    "Enterprise Annual plans may be cancelled within 14 calendar days of purchase.",
    "Support is available 24x7 for Enterprise customers via chat and phone.",
    "Invoices are issued on the first business day of each month.",
]

_vec  = TfidfVectorizer().fit(mini_docs)
_dmat = _vec.transform(mini_docs)

def mini_rag(question, k=1):
    qv     = _vec.transform([question])                  # 1. embed the query
    scores = cosine_similarity(qv, _dmat)[0]             # 2. compare to the index
    top    = np.argsort(scores)[::-1][:k]                # 3. select evidence
    ctx    = "\n".join(f"[S{i+1}] {mini_docs[j]}" for i, j in enumerate(top))
    return ctx, build_prompt(question, [ctx])            # 4. build the context

context, prompt = mini_rag("how long do I have to cancel?")
print("RETRIEVED EVIDENCE:\n" + context)
print("\nThat context is the entire difference between a guess and an answer.")

RETRIEVED EVIDENCE:
[S1] Invoices are issued on the first business day of each month.

That context is the entire difference between a guess and an answer.


In [33]:
_vec

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [35]:
# how to convert sparse matrix to dense matrix
_dmat.toarray()

array([[0.29938511, 0.        , 0.        , 0.29938511, 0.        ,
        0.        , 0.29938511, 0.        , 0.29938511, 0.29938511,
        0.        , 0.        , 0.        , 0.29938511, 0.        ,
        0.22769009, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.29938511, 0.        , 0.22769009, 0.        ,
        0.        , 0.29938511, 0.29938511, 0.        , 0.        ,
        0.        , 0.29938511],
       [0.        , 0.30746099, 0.30746099, 0.        , 0.        ,
        0.30746099, 0.        , 0.        , 0.        , 0.        ,
        0.30746099, 0.30746099, 0.        , 0.        , 0.        ,
        0.23383201, 0.        , 0.30746099, 0.        , 0.30746099,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.30746099, 0.        , 0.        , 0.30746099, 0.        ,
        0.30746099, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.30746099,
        0.        , 0.        , 0.30746099, 0.    